In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Connect and Access Git Repository

import sys
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"



token = getpass("GitHub token: ")


REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())


GitHub token: ··········
Cloning into '/content/bachelor-thesis'...
remote: Enumerating objects: 390, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 390 (delta 63), reused 49 (delta 19), pack-reused 264 (from 1)
Receiving objects: 100% (390/390), 23.95 MiB | 15.94 MiB/s, done.
Resolving deltas: 100% (180/180), done.
/content/bachelor-thesis
Repo exists: True


In [3]:
!pip install -q shap

In [4]:
# Imports
import os
import numpy as np
import torch
import shap



# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.seed import set_seed, RANDOM_SEED
from models.anoformer import AnoFormerMainModel
from explainability.anomaly_score_wrappers import AnoFormerAnomalyScoreWrapper


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [ ]:
# Paths

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

MODEL_DIR = RESULTS_DIR / "models"
MODEL_PATH = MODEL_DIR / "anoformer_40epochs_larger_capacity.pt"

SHAP_RESULTS_DIR = RESULTS_DIR / "shap"
SHAP_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXPLANATION_SAMPLES_PATH = RESULTS_DIR / "evaluation" / "shared_explanation_samples.npz"


print("MODEL_PATH:", MODEL_PATH)
print("SHAP results:", SHAP_RESULTS_DIR)
print("Shared explanation samples:", EXPLANATION_SAMPLES_PATH)

MODEL_PATH: /content/drive/MyDrive/ECG data/results/models/anoformer_40epochs_larger_capacity.pt
SHAP results: /content/drive/MyDrive/ECG data/results/shap


In [6]:
# Load my preprocessed data

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_val = np.load(PROCESSED_DIR / "X_val.npy")
y_val = np.load(PROCESSED_DIR / "y_val.npy")

X_test = np.load(PROCESSED_DIR / "X_test.npy")
y_test = np.load(PROCESSED_DIR / "y_test.npy")

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2183, 1000, 12) (2183,)
(2198, 1000, 12) (2198,)


In [7]:
AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

In [8]:
# Load the pre-trained model

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}"
    )

AnoFormer.Load(str(MODEL_PATH))
AnoFormer.Generator.to(DEVICE)
AnoFormer.Generator.eval()

print("AnoFormer checkpoint loaded.")

Modell geladen von: /content/drive/MyDrive/ECG data/results/models/anoformer_40epochs_larger_capacity.pt
AnoFormer checkpoint loaded.


In [9]:
# Create the Score Wrapper

ScoreWrapper = AnoFormerAnomalyScoreWrapper(
    anoformer_model=AnoFormer,
    window_size=1000,
    masking_rate=None,  # Uses the rate stored in AnoFormer.Masking
    num_masks=1, #try 3 later
    seed=RANDOM_SEED,
).to(DEVICE)

ScoreWrapper.eval()

AnoFormerAnomalyScoreWrapper(
  (generator): TransformerGenerator(
    (InputEmbedding): InputEmbeddingLayer(
      (ProjectionLayer): Linear(in_features=12, out_features=64, bias=True)
      (ActivationReLU): ReLU()
    )
    (PositionalEncoding): PositionalEncoding(
      (DropoutLayer): Dropout(p=0.1, inplace=False)
    )
    (EncoderStack): TransformerEncoderStack(
      (EncoderBlockList): ModuleList(
        (0-1): 2 x TransformerEncoderBlock(
          (SelfAttention): MultiHeadAttention(
            (LinearLayerQuery): Linear(in_features=64, out_features=64, bias=False)
            (LinearLayerKey): Linear(in_features=64, out_features=64, bias=False)
            (LinearLayerValue): Linear(in_features=64, out_features=64, bias=False)
            (LinearLayerOutput): Linear(in_features=64, out_features=64, bias=True)
            (AttentionDropoutLayer): Dropout(p=0.1, inplace=False)
          )
          (NormalizationOne): LayerNormalization()
          (ForwardNetworkLayer): Fo

In [21]:
# Select Background ECG samples from training set for SHAP

BACKGROUND_SIZE = 20

normal_train_indices = np.where(y_train == 0)[0]

rng = np.random.default_rng(RANDOM_SEED)

background_indices = rng.choice(
    normal_train_indices,
    size=BACKGROUND_SIZE,
    replace=False,
)

background_ECGs = torch.as_tensor(
    X_train[background_indices],
    dtype=torch.float32,
    device=DEVICE,
)

print("Background shape:", background_ECGs.shape)
print("Background labels:", y_train[background_indices])

Background shape: torch.Size([20, 1000, 12])
Background labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [22]:
# Save the background indices so the same samples can be used for TranAD SHAP explaination too

np.save(
    SHAP_RESULTS_DIR / "small_background_indices.npy",
    background_indices,
)

In [ ]:
# select ECG samples to explain with SHAP
# I choose the same two samples I used in the Anoformer Attention explanation experiment

import json


with open(EXPLANATION_SAMPLES_PATH, "r") as file:
    selected_samples = json.load(file)

NormalIndex = selected_samples["normal_index"]
AnomalousIndex = selected_samples["anomalous_index"]

print("Normal index:", NormalIndex)
print("Anomalous index:", AnomalousIndex)

Normal index: 2044
Anomalous index: 1801


In [24]:
sample_indices = np.array(
    [NormalIndex, AnomalousIndex]
)

sample_ECGs = torch.as_tensor(
    X_test[sample_indices],
    dtype=torch.float32,
    device=DEVICE,
)

sample_Labels = y_test[sample_indices]

print("ECG shape:", sample_ECGs.shape)
print("Labels:", sample_Labels)

ECG shape: torch.Size([2, 1000, 12])
Labels: [0 1]


In [25]:
ScoreWrapper.eval()

with torch.no_grad():
    ScoresFirstRun = ScoreWrapper(sample_ECGs)
    ScoresSecondRun = ScoreWrapper(sample_ECGs)

print("Scores first run:")
print(ScoresFirstRun)

print("Scores second run:")
print(ScoresSecondRun)

print(
    "Deterministic:",
    torch.allclose(
        ScoresFirstRun,
        ScoresSecondRun,
        atol=1e-7,
    ),
)

Scores first run:
tensor([[0.0136],
        [0.4914]], device='cuda:0')
Scores second run:
tensor([[0.0136],
        [0.4914]], device='cuda:0')
Deterministic: True


In [26]:
with torch.no_grad():
    BackgroundScores = ScoreWrapper(background_ECGs)
    ExplainedScores = ScoreWrapper(sample_ECGs)

print("Mean background score:", BackgroundScores.mean().item())
print("Normal ECG score:", ExplainedScores[0].item())
print("Anomalous ECG score:", ExplainedScores[1].item())

Mean background score: 0.03632169961929321
Normal ECG score: 0.01361917331814766
Anomalous ECG score: 0.4914448857307434


In [30]:
# create a GradientSHAPEplainer and calculate the SHAP values

GradientSHAPExplainer = shap.GradientExplainer(
    ScoreWrapper,
    background_ECGs,
)

SHAPValues = GradientSHAPExplainer.shap_values(
    sample_ECGs
)

/usr/local/lib/python3.12/dist-packages/shap/explainers/_deep/deep_pytorch.py:255: UserWarning: unrecognized nn.Module: LayerNormalization
  warnings.warn(f"unrecognized nn.Module: {module_type}")
